In [33]:
DATA_PATH = "../data/raw/TMDB_movie_dataset_v11.csv"

In [34]:
import os

print(os.getcwd())

c:\Users\DELLA\OneDrive\Documents\ANALISIS BIG DATA\tugas1-abd-2026-main\tugas1-abd-2026-main\notebooks


In [35]:
%pip install polars duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import polars as pl

df = pl.read_csv(DATA_PATH, n_rows=10000)

print(df.head())
print("Shape:", df.shape)

shape: (5, 24)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id     ┆ title      ┆ vote_avera ┆ vote_coun ┆ … ┆ productio ┆ productio ┆ spoken_la ┆ keywords  │
│ ---    ┆ ---        ┆ ge         ┆ t         ┆   ┆ n_compani ┆ n_countri ┆ nguages   ┆ ---       │
│ i64    ┆ str        ┆ ---        ┆ ---       ┆   ┆ es        ┆ es        ┆ ---       ┆ str       │
│        ┆            ┆ f64        ┆ i64       ┆   ┆ ---       ┆ ---       ┆ str       ┆           │
│        ┆            ┆            ┆           ┆   ┆ str       ┆ str       ┆           ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 27205  ┆ Inception  ┆ 8.364      ┆ 34495     ┆ … ┆ Legendary ┆ United    ┆ English,  ┆ rescue,   │
│        ┆            ┆            ┆           ┆   ┆ Pictures, ┆ Kingdom,  ┆ French,   ┆ mission,  │
│        ┆            ┆            ┆           ┆   ┆ Syncopy,  ┆ United    ┆

In [37]:
# Membaca seluruh dataset dengan Polars
df_full = pl.read_csv(
    DATA_PATH,
    null_values=["", "NA", "N/A", "NaN", "null", "NULL"]
)

print("Jumlah baris :", df_full.height)
print("Jumlah kolom :", df_full.width)

Jumlah baris : 1164598
Jumlah kolom : 24


In [38]:
# Melihat struktur dan tipe data dataset
print("=== STRUKTUR DATASET ===")

for column, dtype in zip(df_full.columns, df_full.dtypes):
    print(f"{column}: {dtype}")

=== STRUKTUR DATASET ===
id: Int64
title: String
vote_average: Float64
vote_count: Int64
status: String
release_date: String
revenue: Int64
runtime: Int64
adult: Boolean
backdrop_path: String
budget: Int64
homepage: String
imdb_id: String
original_language: String
original_title: String
overview: String
popularity: Float64
poster_path: String
tagline: String
genres: String
production_companies: String
production_countries: String
spoken_languages: String
keywords: String


In [39]:
# Menghitung jumlah missing value pada setiap kolom
missing_values = df_full.null_count()

print(missing_values)

shape: (1, 24)
┌─────┬───────┬──────────────┬────────────┬───┬─────────────┬─────────────┬─────────────┬──────────┐
│ id  ┆ title ┆ vote_average ┆ vote_count ┆ … ┆ production_ ┆ production_ ┆ spoken_lang ┆ keywords │
│ --- ┆ ---   ┆ ---          ┆ ---        ┆   ┆ companies   ┆ countries   ┆ uages       ┆ ---      │
│ u32 ┆ u32   ┆ u32          ┆ u32        ┆   ┆ ---         ┆ ---         ┆ ---         ┆ u32      │
│     ┆       ┆              ┆            ┆   ┆ u32         ┆ u32         ┆ u32         ┆          │
╞═════╪═══════╪══════════════╪════════════╪═══╪═════════════╪═════════════╪═════════════╪══════════╡
│ 0   ┆ 13    ┆ 0            ┆ 0          ┆ … ┆ 641725      ┆ 521540      ┆ 501715      ┆ 852365   │
└─────┴───────┴──────────────┴────────────┴───┴─────────────┴─────────────┴─────────────┴──────────┘


In [40]:
# Menghitung persentase missing value setiap kolom
missing_percentage = (
    df_full.null_count()
    .transpose(
        include_header=True,
        header_name="Column",
        column_names=["Missing Value"]
    )
    .with_columns(
        (pl.col("Missing Value") / df_full.height * 100)
        .round(2)
        .alias("Missing (%)")
    )
    .sort("Missing (%)", descending=True)
)

missing_percentage

Column,Missing Value,Missing (%)
str,u32,f64
"""homepage""",1041847,89.46
"""tagline""",1001414,85.99
"""backdrop_path""",857312,73.61
"""keywords""",852365,73.19
"""production_companies""",641725,55.1
…,…,…
"""adult""",0,0.0
"""budget""",0,0.0
"""original_language""",0,0.0


In [42]:
# Menghitung jumlah baris duplikat
duplicate_count = df_full.height - df_full.unique().height

print("Jumlah data duplikat:", duplicate_count)

Jumlah data duplikat: 370


In [43]:
# Statistik deskriptif dataset
df_full.describe()

statistic,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
str,f64,str,f64,f64,str,str,f64,f64,f64,str,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str
"""count""",1.164598e6,"""1164585""",1.164598e6,1.164598e6,"""1164598""","""965515""",1.164598e6,1.164598e6,1.164598e6,"""307286""",1.164598e6,"""122751""","""609764""","""1164598""","""1164585""","""924396""",1.164598e6,"""791849""","""163184""","""692684""","""522873""","""643058""","""662883""","""312233"""
"""null_count""",0.0,"""13""",0.0,0.0,"""0""","""199083""",0.0,0.0,0.0,"""857312""",0.0,"""1041847""","""554834""","""0""","""13""","""240202""",0.0,"""372749""","""1001414""","""471914""","""641725""","""521540""","""501715""","""852365"""
"""mean""",756600.65372,null,1.840551,18.418174,null,null,688683.970666,47.352659,0.094549,null,265538.811819,null,null,null,null,null,1.216639,null,null,null,null,null,null,null
"""std""",404105.378466,null,3.003731,314.677114,null,null,1.8527e7,61.61473,null,null,5.0920e6,null,null,null,null,null,7.496802,null,null,null,null,null,null,null
"""min""",2.0,""" """,0.0,0.0,"""Canceled""","""1800-01-01""",-12.0,-28.0,0.0,"""/100IFmS7Pz1vq3BaIB9arYbWma8.j…",0.0,""" ""","""None""","""aa""","""!""",""" """,0.0,"""/1001ZqFYqT5nmP4gYum8X87suZo.j…","""!""","""Action""","""""A ME AND YOU MOTION PICTURE""""","""Afghanistan""","""Abkhazian""","""""on the ""brink of death"""""
"""25%""",419349.0,null,0.0,0.0,null,null,0.0,0.0,null,null,0.0,null,null,null,null,null,0.6,null,null,null,null,null,null,null
"""50%""",762433.0,null,0.0,0.0,null,null,0.0,22.0,null,null,0.0,null,null,null,null,null,0.6,null,null,null,null,null,null,null
"""75%""",1.117169e6,null,4.5,1.0,null,null,0.0,88.0,null,null,0.0,null,null,null,null,null,0.874,null,null,null,null,null,null,null
"""max""",1.420551e6,"""😤 insect james 😤""",10.0,34495.0,"""Rumored""","""2099-11-18""",5.0000e9,14400.0,1.0,"""None""",9.99999999e8,"""https://zyrko0.wixsite.com/wha…","""tt9999999""","""zu""","""𠝹女艷遇""","""😈 Devil and the other""",2994.357,"""None""","""😨""","""Western, War, Thriller""","""ＶＩＬＩＳ ＩＮＣ, Shintoho Pictures""","""Zimbabwe, Spain, South Africa,…","""Zulu, English, Afrikaans""","""회귀, 코미디, 학원"""


In [44]:
# Menghitung jumlah nilai unik pada setiap kolom
unique_values = (
    df_full
    .select(pl.all().n_unique())
    .transpose(
        include_header=True,
        header_name="Column",
        column_names=["Unique Values"]
    )
    .sort("Unique Values")
)

unique_values

Column,Unique Values
str,u32
"""adult""",2
"""status""",6
"""original_language""",174
"""runtime""",763
"""vote_count""",3598
…,…
"""poster_path""",786931
"""overview""",897928
"""title""",997869


In [45]:
# Mengecek nilai negatif pada revenue dan runtime

negative_revenue = df_full.filter(
    pl.col("revenue") < 0
)

negative_runtime = df_full.filter(
    pl.col("runtime") < 0
)

print("Jumlah revenue negatif :", negative_revenue.height)
print("Jumlah runtime negatif :", negative_runtime.height)

Jumlah revenue negatif : 1
Jumlah runtime negatif : 1


In [46]:
# Menampilkan data dengan revenue negatif dan runtime negatif

print("=== Revenue Negatif ===")
print(
    negative_revenue.select([
        "id",
        "title",
        "revenue",
        "runtime",
        "budget",
        "release_date"
    ])
)

print("\n=== Runtime Negatif ===")
print(
    negative_runtime.select([
        "id",
        "title",
        "revenue",
        "runtime",
        "budget",
        "release_date"
    ])
)

=== Revenue Negatif ===
shape: (1, 6)
┌────────┬────────────┬─────────┬─────────┬────────┬──────────────┐
│ id     ┆ title      ┆ revenue ┆ runtime ┆ budget ┆ release_date │
│ ---    ┆ ---        ┆ ---     ┆ ---     ┆ ---    ┆ ---          │
│ i64    ┆ str        ┆ i64     ┆ i64     ┆ i64    ┆ str          │
╞════════╪════════════╪═════════╪═════════╪════════╪══════════════╡
│ 339294 ┆ Some Bears ┆ -12     ┆ 8       ┆ 12     ┆ 2013-05-15   │
└────────┴────────────┴─────────┴─────────┴────────┴──────────────┘

=== Runtime Negatif ===
shape: (1, 6)
┌─────────┬───────────────────┬─────────┬─────────┬────────┬──────────────┐
│ id      ┆ title             ┆ revenue ┆ runtime ┆ budget ┆ release_date │
│ ---     ┆ ---               ┆ ---     ┆ ---     ┆ ---    ┆ ---          │
│ i64     ┆ str               ┆ i64     ┆ i64     ┆ i64    ┆ str          │
╞═════════╪═══════════════════╪═════════╪═════════╪════════╪══════════════╡
│ 1238160 ┆ An Eye For An Eye ┆ 0       ┆ -28     ┆ 0      ┆ null  

In [47]:
# Ringkasan hasil data profiling

total_missing = (
    df_full
    .select(pl.all().null_count())
    .row(0)
)

total_missing_count = sum(total_missing)

print("=== RINGKASAN DATA PROFILING ===")
print(f"Jumlah baris           : {df_full.height:,}")
print(f"Jumlah kolom           : {df_full.width}")
print(f"Jumlah data duplikat   : {duplicate_count:,}")
print(f"Jumlah revenue negatif : {negative_revenue.height}")
print(f"Jumlah runtime negatif : {negative_runtime.height}")
print(f"Total missing value    : {total_missing_count:,}")

=== RINGKASAN DATA PROFILING ===
Jumlah baris           : 1,164,598
Jumlah kolom           : 24
Jumlah data duplikat   : 370
Jumlah revenue negatif : 1
Jumlah runtime negatif : 1
Total missing value    : 7,256,726


In [48]:
import duckdb

# Analitik dataset menggunakan DuckDB SQL
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS unique_ids,
    AVG(vote_average) AS avg_vote_average,
    MAX(vote_count) AS max_vote_count,
    MAX(revenue) AS max_revenue,
    MAX(runtime) AS max_runtime,
    MAX(popularity) AS max_popularity,
    COUNT(*) FILTER (WHERE revenue < 0) AS negative_revenue,
    COUNT(*) FILTER (WHERE runtime < 0) AS negative_runtime
FROM read_csv_auto('{DATA_PATH}')
"""

result = duckdb.sql(query)

result

┌────────────┬────────────┬────────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬──────────────────┬──────────────────┐
│ total_rows │ unique_ids │  avg_vote_average  │ max_vote_count │ max_revenue │ max_runtime │ max_popularity │ negative_revenue │ negative_runtime │
│   int64    │   int64    │       double       │     int64      │    int64    │    int64    │     double     │      int64       │      int64       │
├────────────┼────────────┼────────────────────┼────────────────┼─────────────┼─────────────┼────────────────┼──────────────────┼──────────────────┤
│    1164598 │    1163786 │ 1.8405509351724803 │          34495 │  4999999999 │       14400 │       2994.357 │                1 │                1 │
└────────────┴────────────┴────────────────────┴────────────────┴─────────────┴─────────────┴────────────────┴──────────────────┴──────────────────┘

## Kesimpulan Data Profiling

Berdasarkan hasil data profiling, dataset TMDB Movies memiliki 1.164.598 baris dan 24 kolom sehingga memenuhi ketentuan ukuran dataset, yaitu lebih dari 1 juta baris.

Hasil profiling menggunakan Polars menunjukkan adanya missing value pada beberapa kolom dengan persentase yang cukup tinggi, terutama homepage, tagline, backdrop_path, dan keywords. Dataset juga memiliki 370 baris yang terdeteksi sebagai duplikasi.

Pemeriksaan nilai anomali menemukan 1 data dengan revenue negatif dan 1 data dengan runtime negatif. Data tersebut dicatat sebagai anomali dan tidak diubah pada tahap profiling.

Analitik menggunakan DuckDB menunjukkan terdapat 1.163.786 ID unik dari 1.164.598 baris. Hal ini menunjukkan adanya ID yang muncul lebih dari satu kali dan perlu diperhatikan pada tahap analisis atau preprocessing berikutnya.

Data mentah tidak dilakukan perubahan selama proses profiling.